### Imports

In [ ]:
import pandas as pd
import os
from testgen.utils import *
from pathlib import Path
from dotenv import load_dotenv

In [ ]:
base_path = Path().cwd()
load_dotenv(override=True)

In [3]:
# read data and rename columns
df = pd.read_excel(base_path / "data/sensor_requirements.xlsx")
df_examples = pd.read_excel(base_path / "data/sensor_examples.xlsx")

# Find Examples

In [ ]:
def get_example_txt(df_examples, N_EXAMPLES=1):

    if N_EXAMPLES > 1:
        indexes_to_drop, examples = get_examples_from_df(
            df_examples.drop(columns=["selected"]), N_EXAMPLES
        )

    if N_EXAMPLES == 1:  # pick pre selected example when N_EXAMPLES = 1
        df_examples_n1 = df_examples[df_examples["selected"] == 1].copy()
        df_examples_n1.drop(columns=["selected"], inplace=True)
        indexes_to_drop, examples = get_examples_from_df(df_examples_n1, 1)

    # join all examples in a text format to add to prompt
    examples_txt = ""

    for e1 in examples.values():
        for e2 in e1:
            examples_txt += f"Requirement: {e2[0]}\n"
            # examples_txt += f"Vector: {e2[1]}\n"
            examples_txt += f"Target Sensor/s: {e2[1]}\n"
            examples_txt += "\n"

    return examples_txt, examples

In [ ]:
examples_txt, examples = get_example_txt(df_examples, 1)

print("\n".join(examples_txt.split("\n")[-10:]))

# Response Format

In [ ]:
from testgen.prompts.Sensors import Sensors
from pydantic import Field, create_model

Sensors_t = Sensors.split("\n")


def clean_sensors_fn(x):
    x = x.split(":")
    x[0] = x[0][: x[0].find("(")]
    return x


Sensors_t = list(map(clean_sensors_fn, Sensors_t))

sensor_attrs = {}

for sensor in Sensors_t:
    sensor_attrs[sensor[0].strip().lower().replace(" ", "_")] = (
        int,
        Field(description=sensor[1].strip()),
    )


VectorFormat = create_model("VectorFormat", **sensor_attrs)

# LLM

In [77]:
from testgen.prompts import SystemPrompt
from testgen.prompts import Sensors
from testgen.prompts import UserPromptBulk

In [ ]:
llm_models = {
    "azure": ["gpt-4o-mini", "gpt-4o"],
    "groq": [
        "llama-3.1-8b-instant",
        "llama3-70b-8192",
    ],
}

endpoint_attrs = {
    "azure": {
        "api_key": os.getenv("AZURE_OPENAI_API_KEY"),
        "api_version": os.getenv("AZURE_API_VERSION"),
        "base_url": os.getenv("AZURE_OPENAI_ENDPOINT"),
    },
    "groq": {
        "api_key": os.getenv("GROQ_API_KEY"),
    },
}

### Get Requirements for multi prediction

In [81]:
N_REQS = 2
SAMPLE_TYPE = "random"

batches = get_batches(df, SAMPLE_TYPE, N_REQS)
batches, req_texts = requirement_text_bulk(batches)

Number of Batches: 68
Number of Instances left: 1


In [82]:
results = []

for batch, req_text in tqdm(zip(batches, req_texts)):
    result = invoke_bulk(model_name, client, batch,
                         SystemPrompt, Sensors, examples_txt, UserPromptBulk, req_text)
    results.append(result)

0it [00:00, ?it/s]

In [83]:
total_number_of_instances = len(results) * N_REQS

accuracy = 0
total_tokens = 0
total_completion_tokens = 0
total_time = 0

for r in results:
    accuracy += sum(r["accuracy"])
    total_tokens += r["total_tokens"]
    total_completion_tokens += r["completion_tokens"]
    total_time += r['response_time']

number_of_reqs = len(results)
accuracy /= total_number_of_instances
avg_time_per_req = round(total_time / total_number_of_instances, 6)
avg_token_per_req = total_tokens / total_number_of_instances
avg_completion_token_per_req = total_completion_tokens / total_number_of_instances

In [84]:
print("number_of_reqs:", number_of_reqs)
print("accuracy:", accuracy)
print("avg_time_per_req:", avg_time_per_req)
print("avg_token_per_req:", avg_token_per_req)
print("avg_completion_token_per_req:", avg_completion_token_per_req)

number_of_reqs: 68
accuracy: 0.8897058823529411
avg_time_per_req: 0.476348
avg_token_per_req: 436.375
avg_completion_token_per_req: 15.5


# Save conversation

In [85]:
# save results
time = dt.now()

results_path = "results/bulk-{bulk}_{model}_n-{examples}_acc-{accuracy}_{time}.json"
results_path = results_path.format(
    model=model_name,
    examples=N_EXAMPLES,
    bulk=N_REQS,
    time=time.strftime('%m.%d.%Y-%H:%M:%S'),
    accuracy=round(accuracy, 3)
)

results_file = base_path / results_path
results_file.parent.mkdir(exist_ok=True)
results_file.touch()

with results_file.open("w") as f:
    json.dump({"accuracy": accuracy,
        "number_of_reqs": number_of_reqs,
        "total_tokens": total_tokens,
        "total_completion_tokens": total_completion_tokens,
        "avg_token_per_req": avg_token_per_req,
        "avg_completion_token_per_req": avg_completion_token_per_req,
        "avg_time_per_req": avg_time_per_req,
        "examples": examples,
        "responses": results}, f, indent=4)